In [1]:
from unsloth import FastVisionModel
from transformers import AutoModel
import os
from huggingface_hub import snapshot_download

# 0. Runtime configuration (initialization only)
MODEL_REPO = "unsloth/DeepSeek-OCR-2"
MODEL_DIR = "deepseek_ocr"
INPUT_FOLDER = "input_images"
OUTPUT_FOLDER = "output_results"

LOAD_IN_4BIT = True
FORCE_MODEL_RELOAD = False
UNSLOTH_FORCE_COMPILE = False
GRADIENT_CHECKPOINTING_MODE = "unsloth"

PROMPT = """<image>
Extract all information from this Thai receipt. Output strictly as a JSON object matching the exact schema provided below.

Follow these strict rules:
1. Do not include markdown formatting (like ```json).
2. Do not include any conversational text.
3. If a text field is not present on the receipt, output null (without quotes).
4. If a number or financial field is not present, output 0.0.
5. If there are no individual product items listed, output an empty array [] for "items".

Schema:
{
  "processed": {
    "invoiceType": "",
    "invoiceBook": "",
    "invoiceID": "",
    "invoiceDate": "",
    "purchaseOrderID": "",
    "issuerName": "",
    "issuerAddress": "",
    "issuerTaxID": "",
    "issuerPhone": "",
    "customerName": "",
    "customerAddress": "",
    "customerTaxID": "",
    "customerPhone": "",
    "items": [
      {
        "itemNo": "",
        "itemCode": "",
        "itemName": "",
        "itemUnit": 0,
        "itemUnitCost": 0.0,
        "itemTotalCost": 0.0
      }
    ],
    "totalCost": 0.0,
    "discount": 0.0,
    "totalCostAfterDiscount": 0.0,
    "vat": 0.0,
    "grandTotal": 0.0
  }
}"""

# 1. Hide initialization warnings
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"

# 2. Initialize model/tokenizer only (no OCR loop here)
if (
    not FORCE_MODEL_RELOAD
    and "model" in globals()
    and "tokenizer" in globals()
    and model is not None
    and tokenizer is not None
):
    print("Reusing existing model/tokenizer from current kernel.")
else:
    snapshot_download(MODEL_REPO, local_dir=MODEL_DIR)
    model, tokenizer = FastVisionModel.from_pretrained(
        f"./{MODEL_DIR}",
        load_in_4bit=LOAD_IN_4BIT,
        auto_model=AutoModel,
        trust_remote_code=True,
        unsloth_force_compile=UNSLOTH_FORCE_COMPILE,
        use_gradient_checkpointing=GRADIENT_CHECKPOINTING_MODE,
    )

# 3. Prepare output path for later inference/testing cells
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Model initialization complete.")
print(f"Model repo: {MODEL_REPO}")
print(f"4-bit load: {LOAD_IN_4BIT}")
print(f"Force compile: {UNSLOTH_FORCE_COMPILE}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0409 09:24:34.106000 22664 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[tensorflow|WARNING]From c:\Users\poohz\anaconda3\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



🦥 Unsloth Zoo will now patch everything to make training faster!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr2 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3080 Ti. Num GPUs = 1. Max memory: 12.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


Unsloth: Deepseekocr2 does not support SDPA - switching to fast eager.
Model initialization complete.
Model repo: unsloth/DeepSeek-OCR-2
4-bit load: True
Force compile: False


## Data Prep for Thai Receipt Fine-Tuning

Use this format for supervised data:
- `train_data/images/*.jpg|*.jpeg|*.png`
- `train_data/labels/*.json` (same filename stem as image)

Each label JSON should contain either:
- the full object with `processed`, or
- just the inner `processed` fields.

The next cell converts your data into DeepSeek-OCR conversation format (`messages`) and reuses `PROMPT` as the training instruction.

In [2]:
import json
from pathlib import Path
from PIL import Image

# Folder layout for local fine-tuning data
TRAIN_IMAGES_DIR = "train_data/images"
TRAIN_LABELS_DIR = "train_data/labels"

TEXT_FIELDS = [
    "invoiceType",
    "invoiceBook",
    "invoiceID",
    "invoiceDate",
    "purchaseOrderID",
    "issuerName",
    "issuerAddress",
    "issuerTaxID",
    "issuerPhone",
    "customerName",
    "customerAddress",
    "customerTaxID",
    "customerPhone",
]

TOTAL_FIELDS = [
    "totalCost",
    "discount",
    "totalCostAfterDiscount",
    "vat",
    "grandTotal",
]


def _as_text_or_none(value):
    if value is None:
        return None
    value = str(value).strip()
    return value if value else None


def _as_float(value):
    try:
        if value is None or value == "":
            return 0.0
        return float(value)
    except (TypeError, ValueError):
        return 0.0


def _as_int(value):
    try:
        if value is None or value == "":
            return 0
        return int(float(value))
    except (TypeError, ValueError):
        return 0


def normalize_item(item):
    item = item if isinstance(item, dict) else {}
    return {
        "itemNo": _as_text_or_none(item.get("itemNo")),
        "itemCode": _as_text_or_none(item.get("itemCode")),
        "itemName": _as_text_or_none(item.get("itemName")),
        "itemUnit": _as_int(item.get("itemUnit")),
        "itemUnitCost": _as_float(item.get("itemUnitCost")),
        "itemTotalCost": _as_float(item.get("itemTotalCost")),
    }


def normalize_receipt_label(raw_label):
    source = raw_label if isinstance(raw_label, dict) else {}
    source = source.get("processed", source)

    processed = {field: _as_text_or_none(source.get(field)) for field in TEXT_FIELDS}

    items = source.get("items")
    if not isinstance(items, list):
        items = []
    processed["items"] = [normalize_item(item) for item in items if isinstance(item, dict)]

    for field in TOTAL_FIELDS:
        processed[field] = _as_float(source.get(field))

    return {"processed": processed}


def build_converted_dataset(
    images_dir=TRAIN_IMAGES_DIR,
    labels_dir=TRAIN_LABELS_DIR,
    instruction_prompt=PROMPT,
):
    images_path = Path(images_dir)
    labels_path = Path(labels_dir)
    valid_exts = {".jpg", ".jpeg", ".png"}

    image_files = sorted([p for p in images_path.glob("*") if p.suffix.lower() in valid_exts])

    converted = []
    missing_labels = []

    for image_path in image_files:
        label_path = labels_path / f"{image_path.stem}.json"
        if not label_path.exists():
            missing_labels.append(image_path.name)
            continue

        with label_path.open("r", encoding="utf-8") as f:
            raw_label = json.load(f)

        normalized_label = normalize_receipt_label(raw_label)
        assistant_text = json.dumps(normalized_label, ensure_ascii=False)

        with Image.open(image_path) as img:
            pil_image = img.convert("RGB")

        converted.append(
            {
                "messages": [
                    {
                        "role": "<|User|>",
                        "content": instruction_prompt,
                        "images": [pil_image],
                    },
                    {
                        "role": "<|Assistant|>",
                        "content": assistant_text,
                    },
                ]
            }
        )

    return converted, missing_labels


converted_dataset, missing_label_images = build_converted_dataset()
train_dataset = converted_dataset

print(f"Prepared {len(converted_dataset)} training samples.")
if missing_label_images:
    print(f"Skipped {len(missing_label_images)} images with missing labels.")
    print("First missing labels:", missing_label_images[:10])

Prepared 245 training samples.


In [3]:
import io
import math
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence

from deepseek_ocr.modeling_deepseekocr2 import (
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)


@dataclass
class DeepSeekOCRDataCollator:
    """Adaptive collator for receipt/invoice extraction.

    It does not force one fixed size for every image.
    Size is selected per image, then tokenized with DeepSeek helpers.
    """

    tokenizer: Any
    model: Any
    image_size: int = 768
    base_size: int = 1024
    crop_mode: bool = True
    auto_resize: bool = True
    max_dynamic_crops: int = 6
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 768,
        base_size: int = 1024,
        crop_mode: bool = True,
        auto_resize: bool = True,
        max_dynamic_crops: int = 6,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.auto_resize = auto_resize
        self.max_dynamic_crops = max_dynamic_crops
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
            normalize=True,
        )
        self.patch_size = 16
        self.downsample_ratio = 4
        self.bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else 0
        self.pad_token_id = tokenizer.pad_token_id
        if self.pad_token_id is None:
            self.pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

    def _to_pil(self, image_data) -> Image.Image:
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        if isinstance(image_data, dict) and "bytes" in image_data:
            return Image.open(io.BytesIO(image_data["bytes"])).convert("RGB")
        if isinstance(image_data, str):
            return Image.open(image_data).convert("RGB")
        raise ValueError(f"Unsupported image format: {type(image_data)}")

    def _choose_sizes(self, image: Image.Image) -> Tuple[int, int, bool]:
        if not self.auto_resize:
            return self.base_size, self.image_size, self.crop_mode

        long_side = max(image.size)

        # DeepSeek encoder supports query sizes mapped from base_size 768 or 1024 only.
        if long_side <= 900:
            return 768, 768, False
        return 1024, 768, True

    def _image_tensors_and_tokens(self, image: Image.Image):
        local_base, local_image_size, use_crops = self._choose_sizes(image)

        images_crop_list = []
        width_crop_num, height_crop_num = 1, 1

        if use_crops and max(image.size) > local_image_size:
            images_crop_raw, crop_ratio = dynamic_preprocess(
                image,
                min_num=2,
                max_num=self.max_dynamic_crops,
                image_size=local_image_size,
                use_thumbnail=False,
            )
            width_crop_num, height_crop_num = crop_ratio

            for crop_img in images_crop_raw:
                images_crop_list.append(self.image_transform(crop_img).to(self.dtype))

        global_view = ImageOps.pad(
            image,
            (local_base, local_base),
            color=tuple(int(x * 255) for x in self.image_transform.mean),
        )
        images_ori = self.image_transform(global_view).to(self.dtype).unsqueeze(0)

        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim=0)
        else:
            images_crop = torch.zeros((1, 3, local_base, local_base), dtype=self.dtype)

        images_spatial_crop = torch.tensor([[width_crop_num, height_crop_num]], dtype=torch.long)

        # IMPORTANT: Must match DeepSeek-OCR infer() token layout exactly.
        num_queries = math.ceil((local_image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((local_base // self.patch_size) / self.downsample_ratio)

        # Global: Q_base^2 + 1 separator
        tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
        tokenized_image += [self.image_token_id]

        # Local crops: (Q_local * w) * (Q_local * h), no row separators
        if width_crop_num > 1 or height_crop_num > 1:
            tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num)) * (
                num_queries * height_crop_num
            )

        return images_ori, images_crop, images_spatial_crop, tokenized_image

    def _tokenize_sample(self, messages: List[Dict], tokenized_image: List[int]):
        if len(messages) < 2:
            raise ValueError("Each sample must contain user and assistant messages.")

        user_text = str(messages[0].get("content", ""))
        if "<image>" not in user_text:
            user_text = f"<image>\n{user_text}"

        assistant_text = str(messages[1].get("content", "")).strip()
        eos_text = self.tokenizer.eos_token if self.tokenizer.eos_token is not None else ""

        text_splits = user_text.split("<image>")

        tokenized_str = [self.bos_id]
        images_seq_mask = [False]

        for i, text_sep in enumerate(text_splits):
            t = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
            tokenized_str.extend(t)
            images_seq_mask.extend([False] * len(t))

            if i < len(text_splits) - 1:
                tokenized_str.extend(tokenized_image)
                images_seq_mask.extend([True] * len(tokenized_image))

        prompt_token_count = len(tokenized_str)

        assistant_payload = f"{assistant_text} {eos_text}".strip()
        assistant_tokens = text_encode(self.tokenizer, assistant_payload, bos=False, eos=False)
        tokenized_str.extend(assistant_tokens)
        images_seq_mask.extend([False] * len(assistant_tokens))

        return (
            torch.tensor(tokenized_str, dtype=torch.long),
            torch.tensor(images_seq_mask, dtype=torch.bool),
            prompt_token_count,
        )

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_data = []

        for feature in features:
            messages = feature["messages"]
            image_list = messages[0].get("images", [])
            if not image_list:
                raise ValueError("No image found in sample user message.")

            pil_image = self._to_pil(image_list[0])
            images_ori, images_crop, images_spatial_crop, tokenized_image = self._image_tensors_and_tokens(pil_image)
            input_ids, images_seq_mask, prompt_token_count = self._tokenize_sample(messages, tokenized_image)

            batch_data.append(
                {
                    "input_ids": input_ids,
                    "images_seq_mask": images_seq_mask,
                    "images_ori": images_ori,
                    "images_crop": images_crop,
                    "images_spatial_crop": images_spatial_crop,
                    "prompt_token_count": prompt_token_count,
                }
            )

        input_ids_list = [item["input_ids"] for item in batch_data]
        images_seq_mask_list = [item["images_seq_mask"] for item in batch_data]
        prompt_token_counts = [item["prompt_token_count"] for item in batch_data]

        input_ids = pad_sequence(
            input_ids_list,
            batch_first=True,
            padding_value=self.pad_token_id,
        )
        images_seq_mask = pad_sequence(
            images_seq_mask_list,
            batch_first=True,
            padding_value=False,
        )

        labels = input_ids.clone()
        labels[labels == self.pad_token_id] = -100
        labels[images_seq_mask] = -100

        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        attention_mask = (input_ids != self.pad_token_id).long()

        images_batch = [(item["images_crop"], item["images_ori"]) for item in batch_data]
        images_spatial_crop = torch.cat([item["images_spatial_crop"] for item in batch_data], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }


print("DeepSeekOCRDataCollator is ready (adaptive receipt/invoice mode).")

DeepSeekOCRDataCollator is ready (adaptive receipt/invoice mode).


In [4]:
import math
import os
import torch
from transformers import Trainer, TrainingArguments, TrainerCallback
from unsloth import is_bf16_supported

# ---- Debug-friendly CUDA error reporting ----
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ---- Training knobs (receipt/invoice tuned) ----
USE_MAX_STEPS = True          # False = epoch-based, True = step-based
NUM_TRAIN_EPOCHS = 3
MAX_TRAIN_STEPS = 60
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
OUTPUT_DIR = "outputs/thai_receipt_lora"
DATALOADER_NUM_WORKERS = 0     # Better stability on Windows notebooks
LOG_EVERY_STEPS = 1            # Force visible logs every N optimizer steps

# Adaptive image preprocessing (NOT fixed to one size)
TRAIN_IMAGE_SIZE = 768
TRAIN_BASE_SIZE = 1024
TRAIN_CROP_MODE = True
AUTO_RESIZE_BY_IMAGE = True
MAX_DYNAMIC_CROPS = 6          # Keep aligned with model's dynamic_preprocess defaults

# ---- Safety checks ----
if "train_dataset" not in globals():
    raise RuntimeError("train_dataset not found. Run Cell 3 (Data Prep) first.")
if len(train_dataset) == 0:
    raise RuntimeError("train_dataset is empty. Add image/label pairs and re-run Cell 3.")
if "DeepSeekOCRDataCollator" not in globals():
    raise RuntimeError("DeepSeekOCRDataCollator not found. Run Cell 4 first.")

# Add LoRA adapters once (skip if already wrapped)
if not hasattr(model, "peft_config"):
    model = FastVisionModel.get_peft_model(
        model,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )

FastVisionModel.for_training(model)

data_collator = DeepSeekOCRDataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=TRAIN_IMAGE_SIZE,
    base_size=TRAIN_BASE_SIZE,
    crop_mode=TRAIN_CROP_MODE,
    auto_resize=AUTO_RESIZE_BY_IMAGE,
    max_dynamic_crops=MAX_DYNAMIC_CROPS,
    train_on_responses_only=True,
)

def _move_batch_to_device(batch, model_ref):
    try:
        device = next(model_ref.parameters()).device
    except StopIteration:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    return {
        "input_ids": batch["input_ids"].to(device),
        "attention_mask": batch["attention_mask"].to(device),
        "labels": batch["labels"].to(device),
        "images": [(crops.to(device), ori.to(device)) for crops, ori in batch["images"]],
        "images_seq_mask": batch["images_seq_mask"].to(device),
        "images_spatial_crop": batch["images_spatial_crop"].to(device),
    }


# ---- Preflight check: catch alignment issues before Trainer loop ----
print("[sanity] running one-sample forward check...", flush=True)
try:
    model.eval()
    sample_batch = data_collator([train_dataset[0]])
    sample_batch = _move_batch_to_device(sample_batch, model)

    with torch.no_grad():
        sample_out = model(**sample_batch)

    sample_loss = float(sample_out.loss.detach().cpu()) if sample_out.loss is not None else float("nan")
    print(f"[sanity] forward pass OK. sample_loss={sample_loss:.6f}", flush=True)
except Exception as e:
    print(f"[sanity] forward check FAILED: {type(e).__name__}: {e}", flush=True)
    raise
finally:
    model.train()


effective_batch_size = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
approx_steps_per_epoch = max(1, math.ceil(len(train_dataset) / effective_batch_size))
planned_steps = MAX_TRAIN_STEPS if USE_MAX_STEPS else approx_steps_per_epoch * NUM_TRAIN_EPOCHS

training_args = dict(
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=5,
    learning_rate=LEARNING_RATE,
    logging_steps=LOG_EVERY_STEPS,
    logging_strategy="steps",
    logging_first_step=True,
    log_level="info",
    disable_tqdm=True,  # Use explicit print logs below so progress is always visible
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=3407,
    fp16=not is_bf16_supported(),
    bf16=is_bf16_supported(),
    output_dir=OUTPUT_DIR,
    report_to="none",
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    remove_unused_columns=False,
)

if USE_MAX_STEPS:
    training_args["max_steps"] = MAX_TRAIN_STEPS
else:
    training_args["num_train_epochs"] = NUM_TRAIN_EPOCHS

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_dataset,
    args=TrainingArguments(**training_args),
)


class NotebookProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        print("[train] started", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        loss = logs.get("loss")
        lr = logs.get("learning_rate")
        if loss is not None:
            if lr is not None:
                print(f"[train] step={state.global_step} loss={loss:.6f} lr={lr:.2e}", flush=True)
            else:
                print(f"[train] step={state.global_step} loss={loss:.6f}", flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        print("[train] finished", flush=True)


trainer.add_callback(NotebookProgressCallback())

print("Trainer is ready.", flush=True)
print(f"Samples: {len(train_dataset)}", flush=True)
print(f"Effective batch size: {effective_batch_size}", flush=True)
print(f"Approx steps/epoch: {approx_steps_per_epoch}", flush=True)
print(f"Planned optimizer steps: {planned_steps}", flush=True)
if USE_MAX_STEPS:
    print(f"Training mode: steps ({MAX_TRAIN_STEPS} max steps)", flush=True)
else:
    print(f"Training mode: epochs ({NUM_TRAIN_EPOCHS} epochs)", flush=True)
print(
    f"Adaptive image mode: auto_resize={AUTO_RESIZE_BY_IMAGE}, "
    f"base_size={TRAIN_BASE_SIZE}, image_size={TRAIN_IMAGE_SIZE}, crop_mode={TRAIN_CROP_MODE}",
    flush=True,
)

# Start training
trainer_stats = trainer.train()

# Save adapter and tokenizer for later import testing
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved model artifacts to:", OUTPUT_DIR, flush=True)
print("Final metrics:", trainer_stats.metrics, flush=True)

Unsloth: Making `model.base_model.model.model` require gradients
[sanity] running one-sample forward check...
[sanity] forward pass OK. sample_loss=1.310866


C:\Users\poohz\AppData\Local\Temp\ipykernel_22664\2236030130.py:137: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer._unsloth___init__`. Use `processing_class` instead.
  trainer = Trainer(
max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend


Trainer is ready.
Samples: 245
Effective batch size: 4
Approx steps/epoch: 62
Planned optimizer steps: 60
Training mode: steps (60 max steps)
Adaptive image mode: auto_resize=True, base_size=1024, image_size=768, crop_mode=True


skipped Embedding(129280, 1280): 157.8125M params
skipped Embedding(144, 896): 157.935546875M params
skipped Embedding(256, 896): 158.154296875M params
skipped: 158.154296875M params
***** Running training *****
  Num examples = 245
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 4
  Gradient Accumulation steps = 4
  Total optimization steps = 60
  Number of trainable parameters = 86,307,840


[train] started


...

Unsloth: Will smartly offload gradients to save VRAM!
{'loss': 1.6752, 'grad_norm': 0.7337316870689392, 'learning_rate': 0.0, 'epoch': 0.0163265306122449}
[train] step=1 loss=1.675200 lr=0.00e+00
{'loss': 1.6764, 'grad_norm': 0.7370474934577942, 'learning_rate': 4e-05, 'epoch': 0.0326530612244898}
[train] step=2 loss=1.676400 lr=4.00e-05
{'loss': 1.6424, 'grad_norm': 0.7207680344581604, 'learning_rate': 8e-05, 'epoch': 0.04897959183673469}
[train] step=3 loss=1.642400 lr=8.00e-05
{'loss': 1.6357, 'grad_norm': 0.7102398872375488, 'learning_rate': 0.00012, 'epoch': 0.0653061224489796}
[train] step=4 loss=1.635700 lr=1.20e-04
{'loss': 1.5388, 'grad_norm': 0.6157057285308838, 'learning_rate': 0.00016, 'epoch': 0.08163265306122448}
[train] step=5 loss=1.538800 lr=1.60e-04
{'loss': 1.3191, 'grad_norm': 0.5499873757362366, 'learning_rate': 0.0002, 'epoch': 0.09795918367346938}
[train] step=6 loss=1.319100 lr=2.00e-04
{'loss': 1.1389, 'grad_norm': 0.5573790669441223, 'learning_rate': 0.0001963

Saving model checkpoint to outputs/thai_receipt_lora\checkpoint-60
loading configuration file ./deepseek_ocr\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "first_k_dense_replace": 1,
  "global_view_pos": "head",
  "head_dim": 0,
  "hidden_act": "silu",
  "hidden_size": 1280,
  "initializer_range": 0.02,
  "intermediate_size": 6848,
  "kv_lora_rank": null,
  "language_config": {
    "architectures": [
      "

{'train_runtime': 868.5068, 'train_samples_per_second': 0.276, 'train_steps_per_second': 0.069, 'train_loss': 0.26226023053362346, 'epoch': 0.9795918367346939}
[train] finished


Saving model checkpoint to outputs/thai_receipt_lora
loading configuration file ./deepseek_ocr\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "first_k_dense_replace": 1,
  "global_view_pos": "head",
  "head_dim": 0,
  "hidden_act": "silu",
  "hidden_size": 1280,
  "initializer_range": 0.02,
  "intermediate_size": 6848,
  "kv_lora_rank": null,
  "language_config": {
    "architectures": [
      "DeepseekV2ForC

Saved model artifacts to: outputs/thai_receipt_lora
Final metrics: {'train_runtime': 868.5068, 'train_samples_per_second': 0.276, 'train_steps_per_second': 0.069, 'train_loss': 0.26226023053362346, 'epoch': 0.9795918367346939}


In [ ]:
import os
import glob
from peft import PeftModel
from unsloth import FastVisionModel
from transformers import AutoModel
from huggingface_hub import snapshot_download

# Import saved fine-tuned model and test on one image
BASE_MODEL_REPO = MODEL_REPO if "MODEL_REPO" in globals() else "unsloth/DeepSeek-OCR-2"
BASE_MODEL_DIR = MODEL_DIR if "MODEL_DIR" in globals() else "deepseek_ocr"
FINETUNED_ROOT = OUTPUT_DIR if "OUTPUT_DIR" in globals() else "outputs/thai_receipt_lora"
TEST_IMAGE_PATH = "train_data/images/receipt_003.jpg"
TEST_OUTPUT_DIR = "output_results/test_imported_model"


def find_latest_adapter(root_dir: str) -> str:
    if os.path.exists(os.path.join(root_dir, "adapter_config.json")):
        return root_dir

    checkpoints = glob.glob(os.path.join(root_dir, "checkpoint-*"))

    def step_num(path: str) -> int:
        name = os.path.basename(path)
        tail = name.split("-")[-1]
        return int(tail) if tail.isdigit() else -1

    checkpoints = sorted(checkpoints, key=step_num, reverse=True)
    for ckpt in checkpoints:
        if os.path.exists(os.path.join(ckpt, "adapter_config.json")):
            return ckpt

    raise FileNotFoundError(
        f"No adapter_config.json found under {root_dir}. Run training cell first."
    )


if not os.path.exists(TEST_IMAGE_PATH):
    raise FileNotFoundError(f"Test image not found: {TEST_IMAGE_PATH}")

adapter_dir = find_latest_adapter(FINETUNED_ROOT)
print(f"Using adapter from: {adapter_dir}")

snapshot_download(BASE_MODEL_REPO, local_dir=BASE_MODEL_DIR)

base_model, test_tokenizer = FastVisionModel.from_pretrained(
    f"./{BASE_MODEL_DIR}",
    load_in_4bit=LOAD_IN_4BIT if "LOAD_IN_4BIT" in globals() else True,
    auto_model=AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=False,
    use_gradient_checkpointing=GRADIENT_CHECKPOINTING_MODE if "GRADIENT_CHECKPOINTING_MODE" in globals() else "unsloth",
)

test_model = PeftModel.from_pretrained(base_model, adapter_dir)
FastVisionModel.for_inference(test_model)
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

inference_prompt = (
    PROMPT
    if "PROMPT" in globals()
    else "<image>\nExtract all information and output strict JSON."
)

_ = test_model.infer(
    test_tokenizer,
    prompt=inference_prompt,
    image_file=TEST_IMAGE_PATH,
    output_path=TEST_OUTPUT_DIR,
    base_size=1024,
    image_size=640,
    crop_mode=True,
    save_results=True,
    test_compress=False,
)

print("Import and test completed.")
print(f"Input image: {TEST_IMAGE_PATH}")
print(f"Result file: {TEST_OUTPUT_DIR}/result.mmd")

Using adapter from: outputs/thai_receipt_lora


loading configuration file ./deepseek_ocr\config.json
loading configuration file ./deepseek_ocr\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "first_k_dense_replace": 1,
  "global_view_pos": "head",
  "head_dim": 0,
  "hidden_act": "silu",
  "hidden_size": 1280,
  "initializer_range": 0.02,
  "intermediate_size": 6848,
  "kv_lora_rank": null,
  "language_config": {
    "architectures": [
      "DeepseekV2For

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr2 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3080 Ti. Num GPUs = 1. Max memory: 12.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


loading configuration file ./deepseek_ocr\config.json
loading configuration file ./deepseek_ocr\config.json
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
Model config DeepseekOCR2Config {
  "architectures": [
    "DeepseekOCR2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "modeling_deepseekocr2.DeepseekOCR2Config",
    "AutoModel": "modeling_deepseekocr2.DeepseekOCR2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 0,
  "candidate_resolutions": [
    [
      1024,
      1024
    ]
  ],
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "first_k_dense_replace": 1,
  "global_view_pos": "head",
  "head_dim": 0,
  "hidden_act": "silu",
  "hidden_size": 1280,
  "initializer_range": 0.02,
  "intermediate_size": 6848,
  "kv_lora_rank": null,
  "language_config": {
    "architectures": [
      "DeepseekV2For

Unsloth: Deepseekocr2 does not support SDPA - switching to fast eager.


target_dtype {target_dtype} is replaced by `CustomDtype.INT4` for 4-bit BnB quantization
Generation config file not found, using a generation config created from the model config.
Could not locate the custom_generate/generate.py inside ./deepseek_ocr.


{"processed": {"invoiceType": "ใบกำกับภาษี/ใบเสร็จรับเงิน", "invoiceBook": null, "invoiceID": "LT67051901", "invoiceDate": "19/05/2567", "purchaseOrderID": null, "issuerName": "บริษัท เอ็น แอนด์ เอ็น สปอร์ต แอนด์ เอ็นจิเนียริ่ง จำกัด (สาขาที่ 2)", "issuerAddress": "123/43 หมู่บ้านพิศาล เอวนิว ลาดกระบัง ถนนฉลองกรุง แขวงลำปลาทิว เขตลาดกระบัง กรุงเทพฯ 10520", "issuerTaxID": "0105561192000", "issuerPhone": "082-916-4252", "customerName": "บริษัท ทีทีที บราเธอร์ส จำกัด", "customerAddress": "571/2 ถนนประชาพัฒนา แขวงทับยาว เขตลาดกระบัง กรุงเทพมหานคร 10520", "customerTaxID": "0105560155470", "customerPhone": null, "items": [{"itemNo": "1", "itemCode": null, "itemName": "dmantis d40 speed75 12pcs", "itemUnit": 0, "itemUnitCost": 399.0, "itemTotalCost": 798.0}, {"itemNo": "2", "itemCode": null, "itemName": "dmantis d45 speed75 12pcs", "itemUnit": 0, "itemUnitCost":
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

Import and test completed.
Input image: test_data/images/test_img1.jpg
Result file: output_results/test_imported_model/result.mmd


In [9]:
# Direct eval-mode test on the same uploaded image (no streamer, returns full text)
ALT_TEST_IMAGE_PATH = "train_data/images/receipt_003.jpg"

if "test_model" not in globals() or "test_tokenizer" not in globals():
    raise RuntimeError("Run Cell 6 first to load test_model and test_tokenizer.")

if not os.path.exists(ALT_TEST_IMAGE_PATH):
    raise FileNotFoundError(f"Image not found: {ALT_TEST_IMAGE_PATH}")

eval_output = test_model.infer(
    test_tokenizer,
    prompt=inference_prompt if "inference_prompt" in globals() else "<image>\nExtract all information and output strict JSON.",
    image_file=ALT_TEST_IMAGE_PATH,
    output_path=TEST_OUTPUT_DIR if "TEST_OUTPUT_DIR" in globals() else "output_results/test_imported_model",
    base_size=1024,
    image_size=768,
    crop_mode=True,
    test_compress=False,
    save_results=False,
    eval_mode=True,
 )

print("Eval-mode inference completed.")
print(f"Image: {ALT_TEST_IMAGE_PATH}")
print("Output:")
print(eval_output)

Eval-mode inference completed.
Image: train_data/images/receipt_003.jpg
Output:
{"processed": {"invoiceType": "ใบกำกับภาษี/ใบเสร็จรับเงิน", "invoiceBook": null, "invoiceID": "LT67051901", "invoiceDate": "19/05/2567", "purchaseOrderID": null, "issuerName": "บริษัท เอ็น แอนด์ เอ็น สปอร์ต แอนด์ เอ็นจิเนียริ่ง จำกัด (สาขาที่ 2)", "issuerAddress": "123/43 หมู่บ้านพิศาล เอวนิว ลาดกระบัง ถนนฉลองกรุง แขวงลำปลาทิว เขตลาดกระบัง กรุงเทพฯ 10520", "issuerTaxID": "0105561192000", "issuerPhone": "082-916-4252", "customerName": "บริษัท ทีทีที บราเธอร์ส จำกัด", "customerAddress": "571/2 ถนนประชาพัฒนา แขวงทับยาว เขตลาดกระบัง กรุงเทพมหานคร 10520", "customerTaxID": "0105560155470", "customerPhone": null, "items": [{"itemNo": "1", "itemCode": null, "itemName": "dmantis d40 speed75 12pcs", "itemUnit": 0, "itemUnitCost": 399.0, "itemTotalCost": 798.0}, {"itemNo": "2", "itemCode": null, "itemName": "dmantis d45 speed75 12pcs", "itemUnit": 0, "itemUnitCost": 449.0, "itemTotalCost": 898.0}, {"itemNo": "3", "ite